# FYDP DeBERTa Baseline (Full Training)
Trains seeds 42, 7, 123. Download the `baseline_deberta_seed*/` folders after completion.

In [ ]:
import os, re, json, random, math
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import torch
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    classification_report, confusion_matrix, roc_curve, auc
)
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import label_binarize
from transformers import (
    AutoModelForSequenceClassification, AutoTokenizer,
    DataCollatorWithPadding, Trainer, TrainingArguments, TrainerCallback
)
os.environ["TOKENIZERS_PARALLELISM"] = "false"

class NanGuardCallback(TrainerCallback):
    def on_step_end(self, args, state, control, **kwargs):
        model = kwargs.get("model")
        if model is None:
            return control
        for name, param in model.named_parameters():
            if param.grad is not None and not torch.isfinite(param.grad).all():
                print(f"[NanGuard] non-finite gradient at global_step={state.global_step}, param={name}")
                control.should_training_stop = True
                return control
        return control

In [ ]:
DATA_PATH = "/kaggle/input/datasets/armadarmard/fydpfinal/final_balanced_8K_curated.csv"
MODEL_NAME = "microsoft/deberta-v3-base"
SEEDS = [42, 7, 123]
MAX_LENGTH = 512
BATCH_SIZE = 8
EPOCHS = 3
LEARNING_RATE = 3e-6
WEIGHT_DECAY = 0.01
WARMUP_RATIO = 0.1
TEST_SIZE = 0.20
VAL_SIZE = 0.10
MAX_GRAD_NORM = 0.3
ADAM_EPSILON = 1e-6
FREEZE_EMBEDDINGS = False

In [ ]:
def set_seed(seed):
    random.seed(seed); np.random.seed(seed)
    torch.manual_seed(seed); torch.cuda.manual_seed_all(seed)

df = pd.read_csv(DATA_PATH)
df = df[["Label", "Text"]].dropna().reset_index(drop=True)
df["Text"] = df["Text"].apply(lambda x: re.sub(r"\s+", " ", str(x).replace("\n"," ")).strip())

labels = sorted(df["Label"].unique())
label2id = {l:i for i,l in enumerate(labels)}
id2label = {i:l for i,l in enumerate(labels)}
df["label_id"] = df["Label"].map(label2id)
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

class TextDataset(torch.utils.data.Dataset):
    def __init__(self, texts, labels, tokenizer, max_len=512):
        self.texts, self.labels, self.tokenizer, self.max_len = list(texts), list(labels), tokenizer, max_len
    def __len__(self): return len(self.labels)
    def __getitem__(self, idx):
        enc = self.tokenizer(str(self.texts[idx]), truncation=True, padding="max_length", max_length=self.max_len)
        enc["labels"] = int(self.labels[idx])
        return enc

print(f"Dataset: {df.shape}, classes: {label2id}, seeds: {SEEDS}")

In [ ]:
results = []
for seed in SEEDS:
    print(f"\n{'='*60}\nSeed {seed}\n{'='*60}")
    set_seed(seed)
    out = Path(f"baseline_deberta_seed{seed}")
    out.mkdir(parents=True, exist_ok=True)

    train_val, test = train_test_split(df, test_size=TEST_SIZE, stratify=df["label_id"], random_state=seed)
    train, val = train_test_split(train_val, test_size=VAL_SIZE, stratify=train_val["label_id"], random_state=seed)
    print(f"Train: {len(train)}, Val: {len(val)}, Test: {len(test)}")

    train_ds = TextDataset(train["Text"], train["label_id"], tokenizer, MAX_LENGTH)
    val_ds = TextDataset(val["Text"], val["label_id"], tokenizer, MAX_LENGTH)
    test_ds = TextDataset(test["Text"], test["label_id"], tokenizer, MAX_LENGTH)

    model = AutoModelForSequenceClassification.from_pretrained(
        MODEL_NAME, num_labels=len(label2id), id2label=id2label, label2id=label2id
    )
    if FREEZE_EMBEDDINGS:
        for p in model.deberta.embeddings.parameters():
            p.requires_grad = False
        print("[freeze_embeddings] frozen.")

    args = TrainingArguments(
        output_dir=str(out / "checkpoints"), seed=seed,
        learning_rate=LEARNING_RATE, adam_epsilon=ADAM_EPSILON,
        per_device_train_batch_size=BATCH_SIZE, per_device_eval_batch_size=BATCH_SIZE,
        num_train_epochs=EPOCHS, weight_decay=WEIGHT_DECAY, warmup_ratio=WARMUP_RATIO,
        eval_strategy="epoch", save_strategy="epoch", save_total_limit=5,
        load_best_model_at_end=True, metric_for_best_model="macro_f1", greater_is_better=True,
        report_to="none", fp16=False, max_grad_norm=MAX_GRAD_NORM,
    )

    def compute_metrics(eval_pred):
        logits, labels = eval_pred
        preds = np.argmax(logits, axis=-1)
        return {"macro_f1": f1_score(labels, preds, average="macro")}

    trainer = Trainer(
        model=model, args=args,
        train_dataset=train_ds, eval_dataset=val_ds,
        processing_class=tokenizer, data_collator=DataCollatorWithPadding(tokenizer=tokenizer),
        compute_metrics=compute_metrics,
        callbacks=[NanGuardCallback()],
    )
    trainer.train()

    pred = trainer.predict(test_ds)
    y_true = pred.label_ids
    y_pred = np.argmax(pred.predictions, axis=-1)
    y_prob = torch.softmax(torch.tensor(pred.predictions), dim=-1).numpy()

    metrics = {
        "accuracy": accuracy_score(y_true, y_pred),
        "macro_f1": f1_score(y_true, y_pred, average="macro"),
        "weighted_f1": f1_score(y_true, y_pred, average="weighted"),
        "precision": precision_score(y_true, y_pred, average="weighted", zero_division=0),
        "recall": recall_score(y_true, y_pred, average="weighted", zero_division=0),
        "seed": seed,
    }
    results.append(metrics)
    pd.DataFrame([metrics]).to_csv(out / "baseline_metrics.csv", index=False)
    report = classification_report(y_true, y_pred, target_names=labels, zero_division=0)
    with open(out / "classification_report.txt", "w") as f:
        f.write(report)
    print(f"\n{report}")

    cm = confusion_matrix(y_true, y_pred)
    plt.figure(figsize=(5,4))
    sns.heatmap(cm, annot=True, fmt="d", cmap="Blues", xticklabels=labels, yticklabels=labels)
    plt.title(f"DeBERTa Baseline (seed={seed})")
    plt.xlabel("Predicted"); plt.ylabel("True")
    plt.tight_layout(); plt.savefig(out / "confusion_matrix.png", dpi=200); plt.close()

    y_bin = label_binarize(y_true, classes=range(len(labels)))
    plt.figure(figsize=(7,6))
    for i, c in enumerate(["#1f77b4","#ff7f0e","#2ca02c","#d62728"]):
        fpr, tpr, _ = roc_curve(y_bin[:,i], y_prob[:,i])
        plt.plot(fpr, tpr, color=c, lw=2, label=f"{labels[i]} (AUC={auc(fpr,tpr):.3f})")
    plt.plot([0,1],[0,1],"k:",lw=1)
    plt.xlim([-0.02,1.02]); plt.ylim([-0.02,1.02])
    plt.xlabel("FPR"); plt.ylabel("TPR")
    plt.title(f"DeBERTa Baseline (seed={seed}) - ROC")
    plt.legend(loc="lower right", fontsize=8); plt.grid(alpha=0.3)
    plt.tight_layout(); plt.savefig(out / "roc.png", dpi=200); plt.close()

    print(f"Seed {seed} done \u2014 macro_f1={metrics['macro_f1']:.4f}")

summary = pd.DataFrame(results)
summary.to_csv("baseline_deberta_summary.csv", index=False)
print(f"\n{'='*60}\nAll seeds complete!")
print(summary)
print("\nDownload each baseline_deberta_seed*/ folder.")